# 🔧 NOC Tune - TTFB Network Quality Test

**Notebook untuk pengujian kualitas jaringan dengan mengukur TTFB (Time To First Byte)**

## ✨ Fitur Auto-Deteksi
Script ini akan **otomatis mendeteksi**:
- 📶 **Kekuatan sinyal WiFi** (dBm) → dikategorikan Good/Bad berdasarkan threshold
- 📻 **Band WiFi** (2.4GHz / 5GHz) → dari channel number
- 🌐 **DNS Server** yang sedang digunakan
- 📍 **Lokasi & ISP** → via IP Geolocation atau Browser GPS

## 🎯 Lokasi GPS Presisi (Opsional)
Untuk mendapatkan lokasi GPS yang lebih akurat (±10m vs ±1km):
```bash
python get_location.py
```
Script akan membuka browser untuk meminta izin lokasi dari user.

## ⚙️ Config File
Edit `config.txt` untuk:
- Target URL yang akan ditest
- Threshold kategorisasi (signal, TTFB)
- ONT DNS (sebagai referensi jika auto-detect gagal)

## 📊 Output
Report otomatis dengan nama file informatif:
```
results/session_good_signal_5G_8-8-8-8_20260413_123456/
├── session_info.json      # Info lengkap termasuk koordinat
├── ping_result.txt
├── Target_1_instagram.com.csv
├── all_results.csv
├── summary.csv
└── analysis_chart.png
```

## 🚀 Cara Penggunaan
1. (Opsional) Jalankan `python get_location.py` untuk lokasi presisi
2. Edit `config.txt` jika diperlukan
3. Klik **Run All** dan tunggu hasil

## 💡 Tips untuk macOS
Untuk deteksi WiFi signal (RSSI) yang lebih baik:
```bash
pip install pyobjc-framework-CoreWLAN
```

---

In [4]:
# ============================================
# CELL 1: IMPORTS & SETUP
# ============================================
import subprocess
import pandas as pd
import numpy as np
from datetime import datetime
import time
import os
import re
import platform
import sys
import json
from pathlib import Path
from urllib.parse import urlparse

print("✅ Libraries loaded")
print(f"🖥️  System: {platform.system()}")
print(f"📅 Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

✅ Libraries loaded
🖥️  System: Darwin
📅 Date: 2026-04-13 11:44:16


In [5]:
# ============================================
# CELL 2: READ & VALIDATE CONFIG
# ============================================

def parse_config(config_path: str) -> dict:
    """
    Parse config.txt file and return configuration dictionary.
    """
    config = {}
    required_keys = ['TARGETS', 'SAMPLE_COUNT', 'DELAY_SECONDS', 'PING_DURATION', 
                     'SIGNAL_THRESHOLD_DBM', 'TTFB_GOOD_MS', 'TTFB_WARNING_MS']
    
    if not os.path.exists(config_path):
        raise FileNotFoundError(f"❌ Config file not found: {config_path}")
    
    with open(config_path, 'r') as f:
        for line_num, line in enumerate(f, 1):
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            if '=' not in line:
                raise ValueError(f"❌ Invalid format at line {line_num}: '{line}'")
            
            key, value = line.split('=', 1)
            key = key.strip()
            value = value.strip()
            if key:
                config[key] = value
    
    # Validate required keys
    missing = [k for k in required_keys if k not in config]
    if missing:
        raise ValueError(f"❌ Missing required config: {', '.join(missing)}")
    
    # Parse values
    targets_raw = config['TARGETS']
    targets = [t.strip() for t in targets_raw.split(',') if t.strip()]
    if not targets:
        raise ValueError("❌ TARGETS cannot be empty")
    for t in targets:
        if not t.startswith('http'):
            raise ValueError(f"❌ Invalid URL format: {t}")
    config['TARGETS'] = targets
    
    config['SAMPLE_COUNT'] = int(config['SAMPLE_COUNT'])
    config['DELAY_SECONDS'] = int(config['DELAY_SECONDS'])
    config['PING_DURATION'] = int(config['PING_DURATION'])
    config['SIGNAL_THRESHOLD_DBM'] = int(config['SIGNAL_THRESHOLD_DBM'])
    config['TTFB_GOOD_MS'] = int(config['TTFB_GOOD_MS'])
    config['TTFB_WARNING_MS'] = int(config['TTFB_WARNING_MS'])
    
    # Optional: ONT_DNS (for reference if auto-detect fails)
    config['ONT_DNS'] = config.get('ONT_DNS', 'N/A')
    
    return config


# Load config
CONFIG_PATH = Path("./config.txt")

try:
    CONFIG = parse_config(CONFIG_PATH)
    print("=" * 60)
    print("📋 KONFIGURASI BERHASIL DIMUAT")
    print("=" * 60)
    print(f"\n🎯 Targets ({len(CONFIG['TARGETS'])})")
    for i, t in enumerate(CONFIG['TARGETS'], 1):
        print(f"   {i}. {t}")
    print(f"\n📊 Parameter Test:")
    print(f"   • Sample count: {CONFIG['SAMPLE_COUNT']}x per target")
    print(f"   • Delay: {CONFIG['DELAY_SECONDS']}s antar test")
    print(f"   • Ping duration: {CONFIG['PING_DURATION']}s")
    print(f"\n📶 Signal Threshold: {CONFIG['SIGNAL_THRESHOLD_DBM']} dBm")
    print(f"   • Bad signal: < {CONFIG['SIGNAL_THRESHOLD_DBM']} dBm")
    print(f"   • Good signal: >= {CONFIG['SIGNAL_THRESHOLD_DBM']} dBm")
    print(f"\n⏱️ TTFB Threshold:")
    print(f"   • Good: < {CONFIG['TTFB_GOOD_MS']}ms")
    print(f"   • Warning: {CONFIG['TTFB_GOOD_MS']}-{CONFIG['TTFB_WARNING_MS']}ms")
    print(f"   • Poor: > {CONFIG['TTFB_WARNING_MS']}ms")
    if CONFIG['ONT_DNS'] != 'N/A':
        print(f"\n🌐 ONT DNS (referensi): {CONFIG['ONT_DNS']}")
    
    total_tests = len(CONFIG['TARGETS']) * CONFIG['SAMPLE_COUNT']
    est_minutes = (total_tests * CONFIG['DELAY_SECONDS'] + CONFIG['PING_DURATION']) // 60
    print(f"\n⏳ Estimasi waktu: ~{est_minutes} menit")
    print("=" * 60)
    
except (FileNotFoundError, ValueError) as e:
    print(str(e))
    print("\n💡 Pastikan file config.txt ada dan formatnya benar.")
    raise SystemExit("Config validation failed")

📋 KONFIGURASI BERHASIL DIMUAT

🎯 Targets (2)
   1. https://www.instagram.com
   2. https://qt-google-cloud-cdn.bronze.systems

📊 Parameter Test:
   • Sample count: 10x per target
   • Delay: 30s antar test
   • Ping duration: 60s

📶 Signal Threshold: -65 dBm
   • Bad signal: < -65 dBm
   • Good signal: >= -65 dBm

⏱️ TTFB Threshold:
   • Good: < 600ms
   • Warning: 600-800ms
   • Poor: > 800ms

⏳ Estimasi waktu: ~11 menit


In [ ]:
# ============================================
# CELL 3: AUTO-DETECT NETWORK CONDITIONS
# ============================================
import urllib.request

# Path for precise location from browser-based geolocation script
PRECISE_LOCATION_FILE = Path("./precise_location.json")

def get_wifi_signal_macos() -> dict:
    """
    Auto-detect WiFi on macOS using multiple fallback methods.
    """
    result = {
        "ssid": None,
        "rssi": None,
        "channel": None,
        "band": None,
        "error": None
    }
    
    # Method 1: Try system_profiler (most reliable on modern macOS)
    try:
        cmd = ["system_profiler", "SPAirPortDataType", "-json"]
        proc = subprocess.run(cmd, capture_output=True, text=True, timeout=15)
        
        if proc.returncode == 0:
            import json
            data = json.loads(proc.stdout)
            
            # Navigate the JSON structure
            airport_data = data.get("SPAirPortDataType", [])
            if airport_data:
                for item in airport_data:
                    interfaces = item.get("spairport_airport_interfaces", [])
                    for iface in interfaces:
                        current_network = iface.get("spairport_current_network_information", {})
                        if current_network:
                            result["ssid"] = current_network.get("_name")
                            
                            # Get PHY Mode to determine band
                            phy_mode = current_network.get("spairport_current_network_information_phy_mode", "")
                            if "ac" in phy_mode.lower() or "ax" in phy_mode.lower() or "5GHz" in phy_mode:
                                result["band"] = "5GHz"
                            elif "n" in phy_mode.lower() or "g" in phy_mode.lower() or "b" in phy_mode.lower():
                                # 802.11n can be both, but b/g is 2.4GHz
                                if "2.4" in phy_mode or "2,4" in phy_mode:
                                    result["band"] = "2.4GHz"
                                elif "5" in phy_mode:
                                    result["band"] = "5GHz"
                            
                            # Get channel
                            channel_str = current_network.get("spairport_current_network_information_channel", "")
                            if channel_str:
                                ch_match = re.search(r'(\d+)', str(channel_str))
                                if ch_match:
                                    ch = int(ch_match.group(1))
                                    result["channel"] = ch
                                    # Determine band from channel if not already set
                                    if result["band"] is None:
                                        result["band"] = "5GHz" if ch >= 36 else "2.4GHz"
                            
                            # RSSI might not be in system_profiler, try wdutil or other methods
                            break
            
            if result["ssid"]:
                # Try to get RSSI using wdutil (requires sudo, may not work)
                # Fallback: use networkQuality or estimate
                pass
                
    except Exception as e:
        pass  # Continue to next method
    
    # Method 2: Try airport at various paths
    if result["ssid"] is None:
        airport_paths = [
            "/System/Library/PrivateFrameworks/Apple80211.framework/Versions/Current/Resources/airport",
            "/System/Library/PrivateFrameworks/Apple80211.framework/Resources/airport",
            "/usr/local/bin/airport"
        ]
        
        for airport_path in airport_paths:
            if os.path.exists(airport_path):
                try:
                    cmd = [airport_path, "-I"]
                    proc = subprocess.run(cmd, capture_output=True, text=True, timeout=10)
                    
                    if proc.returncode == 0:
                        output = proc.stdout
                        
                        match = re.search(r'\s+SSID: (.+)', output)
                        if match:
                            result["ssid"] = match.group(1).strip()
                        
                        match = re.search(r'\s+agrCtlRSSI: (-?\d+)', output)
                        if match:
                            result["rssi"] = int(match.group(1))
                        
                        match = re.search(r'\s+channel: (\d+)', output)
                        if match:
                            ch = int(match.group(1))
                            result["channel"] = ch
                            result["band"] = "5GHz" if ch >= 36 else "2.4GHz"
                        break
                except:
                    continue
    
    # Method 3: Use networksetup to at least get SSID
    if result["ssid"] is None:
        try:
            cmd = ["networksetup", "-getairportnetwork", "en0"]
            proc = subprocess.run(cmd, capture_output=True, text=True, timeout=10)
            
            if proc.returncode == 0:
                output = proc.stdout
                match = re.search(r'Current Wi-Fi Network: (.+)', output)
                if match:
                    result["ssid"] = match.group(1).strip()
        except:
            pass
    
    # Method 4: Try using CoreWLAN via Python (if pyobjc is installed)
    if result["ssid"] is None or result["rssi"] is None:
        try:
            import objc
            from CoreWLAN import CWInterface, CWWiFiClient
            
            client = CWWiFiClient.sharedWiFiClient()
            interface = client.interface()
            
            if interface:
                result["ssid"] = interface.ssid()
                result["rssi"] = interface.rssiValue()
                ch = interface.wlanChannel()
                if ch:
                    result["channel"] = ch.channelNumber()
                    result["band"] = "5GHz" if result["channel"] >= 36 else "2.4GHz"
        except ImportError:
            pass  # pyobjc not installed
        except Exception:
            pass
    
    if result["ssid"] is None and result["rssi"] is None:
        result["error"] = "Could not detect WiFi (try: pip install pyobjc-framework-CoreWLAN)"
    
    return result


def get_wifi_signal() -> dict:
    """
    Auto-detect WiFi signal strength, band, and channel.
    """
    system = platform.system()
    result = {
        "ssid": None,
        "rssi": None,
        "channel": None,
        "band": None,
        "error": None
    }
    
    try:
        if system == "Darwin":  # macOS
            result = get_wifi_signal_macos()
                    
        elif system == "Windows":
            cmd = ["netsh", "wlan", "show", "interfaces"]
            proc = subprocess.run(cmd, capture_output=True, text=True, timeout=10)
            
            if proc.returncode == 0:
                output = proc.stdout
                
                match = re.search(r'SSID\s+:\s+(.+)', output)
                if match:
                    result["ssid"] = match.group(1).strip()
                
                match = re.search(r'Signal\s+:\s+(\d+)%', output)
                if match:
                    signal_pct = int(match.group(1))
                    result["rssi"] = int(-100 + (signal_pct * 0.7))
                
                match = re.search(r'Channel\s+:\s+(\d+)', output)
                if match:
                    ch = int(match.group(1))
                    result["channel"] = ch
                    result["band"] = "5GHz" if ch >= 36 else "2.4GHz"
                    
        elif system == "Linux":
            # Try nmcli first (more reliable)
            try:
                cmd = ["nmcli", "-t", "-f", "active,ssid,signal,chan", "dev", "wifi"]
                proc = subprocess.run(cmd, capture_output=True, text=True, timeout=10)
                
                if proc.returncode == 0:
                    for line in proc.stdout.strip().split('\n'):
                        parts = line.split(':')
                        if len(parts) >= 4 and parts[0] == 'yes':
                            result["ssid"] = parts[1]
                            signal_pct = int(parts[2]) if parts[2] else None
                            if signal_pct:
                                result["rssi"] = int(-100 + (signal_pct * 0.7))
                            ch = int(parts[3]) if parts[3] else None
                            if ch:
                                result["channel"] = ch
                                result["band"] = "5GHz" if ch >= 36 else "2.4GHz"
                            break
            except:
                pass
            
            # Fallback to iwconfig
            if result["ssid"] is None:
                cmd = ["iwconfig"]
                proc = subprocess.run(cmd, capture_output=True, text=True, timeout=10)
                
                if proc.returncode == 0:
                    output = proc.stdout
                    
                    match = re.search(r'ESSID:\"(.+?)\"', output)
                    if match:
                        result["ssid"] = match.group(1)
                    
                    match = re.search(r'Signal level[=:]\s*(-?\d+)', output)
                    if match:
                        result["rssi"] = int(match.group(1))
                    
        if result["rssi"] is None and result["ssid"] is None:
            if not result["error"]:
                result["error"] = "Could not detect WiFi signal"
            
    except subprocess.TimeoutExpired:
        result["error"] = "Timeout while checking WiFi"
    except Exception as e:
        result["error"] = str(e)
    
    return result


def get_dns_servers() -> dict:
    """
    Auto-detect current DNS servers.
    """
    system = platform.system()
    result = {
        "dns_servers": [],
        "primary_dns": None,
        "error": None
    }
    
    try:
        if system == "Darwin":  # macOS
            # Try scutil --dns first (more reliable)
            cmd = ["scutil", "--dns"]
            proc = subprocess.run(cmd, capture_output=True, text=True, timeout=10)
            
            if proc.returncode == 0:
                output = proc.stdout
                matches = re.findall(r'nameserver\[\d+\]\s*:\s*(\d+\.\d+\.\d+\.\d+)', output)
                if matches:
                    result["dns_servers"] = list(dict.fromkeys(matches))[:4]
                    result["primary_dns"] = result["dns_servers"][0]
            
            # Fallback to networksetup
            if not result["dns_servers"]:
                cmd = ["networksetup", "-getdnsservers", "Wi-Fi"]
                proc = subprocess.run(cmd, capture_output=True, text=True, timeout=10)
                
                if proc.returncode == 0:
                    output = proc.stdout.strip()
                    if "There aren't any DNS Servers set" not in output:
                        dns_list = [line.strip() for line in output.split('\n') 
                                   if re.match(r'^\d+\.\d+\.\d+\.\d+$', line.strip())]
                        if dns_list:
                            result["dns_servers"] = dns_list
                            result["primary_dns"] = dns_list[0]
                        
        elif system == "Windows":
            cmd = ["ipconfig", "/all"]
            proc = subprocess.run(cmd, capture_output=True, text=True, timeout=10)
            
            if proc.returncode == 0:
                output = proc.stdout
                matches = re.findall(r'DNS Servers[\s.]*:\s*(\d+\.\d+\.\d+\.\d+)', output)
                if matches:
                    result["dns_servers"] = list(dict.fromkeys(matches))[:4]
                    result["primary_dns"] = result["dns_servers"][0]
                    
        elif system == "Linux":
            if os.path.exists("/etc/resolv.conf"):
                with open("/etc/resolv.conf", "r") as f:
                    content = f.read()
                    matches = re.findall(r'nameserver\s+(\d+\.\d+\.\d+\.\d+)', content)
                    if matches:
                        result["dns_servers"] = matches[:4]
                        result["primary_dns"] = result["dns_servers"][0]
                        
        if not result["dns_servers"]:
            result["error"] = "Could not detect DNS servers"
            
    except Exception as e:
        result["error"] = str(e)
    
    return result


def get_device_location() -> dict:
    """
    Get device location using multiple methods:
    1. Check for precise location file (from browser-based get_location.py script)
    2. Fallback to IP geolocation API
    
    For precise GPS location, run: python get_location.py
    """
    result = {
        "latitude": None,
        "longitude": None,
        "accuracy": None,
        "city": None,
        "region": None,
        "country": None,
        "isp": None,
        "ip": None,
        "method": None,
        "error": None
    }
    
    # Method 1: Check for precise location file from browser-based script
    if PRECISE_LOCATION_FILE.exists():
        try:
            with open(PRECISE_LOCATION_FILE, 'r') as f:
                precise_data = json.load(f)
            
            # Check if data is recent (within last 24 hours)
            saved_at = precise_data.get('saved_at')
            if saved_at:
                from datetime import datetime, timedelta
                try:
                    saved_time = datetime.fromisoformat(saved_at.replace('Z', '+00:00'))
                    if datetime.now().astimezone() - saved_time < timedelta(hours=24):
                        result["latitude"] = precise_data.get("latitude")
                        result["longitude"] = precise_data.get("longitude")
                        result["accuracy"] = precise_data.get("accuracy")
                        result["method"] = f"Browser Geolocation (±{precise_data.get('accuracy', '?')}m)"
                        print(f"   📍 Using precise location from browser (saved {saved_at})")
                except:
                    pass
        except Exception as e:
            pass  # Continue to fallback methods
    
    # Method 2: Try CoreWLAN positioning on macOS (if available)
    if result["latitude"] is None and platform.system() == "Darwin":
        try:
            import objc
            from CoreLocation import CLLocationManager
            # Note: CoreLocation requires user permission and event loop
            # Better to use the browser-based approach
            pass
        except ImportError:
            pass
        except Exception:
            pass
    
    # Method 3: IP Geolocation API (works without permission, less accurate)
    try:
        # Using ip-api.com (free, no API key needed)
        url = "http://ip-api.com/json/?fields=status,message,country,regionName,city,lat,lon,isp,query"
        
        req = urllib.request.Request(url, headers={'User-Agent': 'NOC-Tune/1.0'})
        with urllib.request.urlopen(req, timeout=10) as response:
            data = json.loads(response.read().decode())
            
            if data.get("status") == "success":
                # Only update if we don't have precise location
                if result["latitude"] is None:
                    result["latitude"] = data.get("lat")
                    result["longitude"] = data.get("lon")
                    result["method"] = "IP Geolocation (~city level)"
                
                # Always get city/country/ISP from IP API
                result["city"] = data.get("city")
                result["region"] = data.get("regionName")
                result["country"] = data.get("country")
                result["isp"] = data.get("isp")
                result["ip"] = data.get("query")
            else:
                if result["latitude"] is None:
                    result["error"] = data.get("message", "IP geolocation failed")
                
    except Exception as e:
        if result["latitude"] is None:
            result["error"] = f"Location detection failed: {str(e)}"
    
    return result


def categorize_signal(rssi: int, threshold: int) -> str:
    """
    Categorize signal strength as good or bad.
    """
    if rssi is None:
        return "unknown_signal"
    return "good_signal" if rssi >= threshold else "bad_signal"


# ==== AUTO-DETECT ALL CONDITIONS ====
print("=" * 60)
print("🔍 AUTO-DETECT KONDISI JARINGAN")
print("=" * 60)

# 1. Detect WiFi Signal
print("\n📶 Mendeteksi WiFi...")
SIGNAL_INFO = get_wifi_signal()

if SIGNAL_INFO["error"]:
    print(f"   ⚠️ Warning: {SIGNAL_INFO['error']}")
    SIGNAL_CATEGORY = "unknown_signal"
    SIGNAL_DISPLAY = "Unknown Signal"
else:
    print(f"   📡 SSID: {SIGNAL_INFO['ssid']}")
    if SIGNAL_INFO['rssi']:
        print(f"   📶 RSSI: {SIGNAL_INFO['rssi']} dBm")
    else:
        print(f"   📶 RSSI: (tidak terdeteksi)")
    if SIGNAL_INFO['channel']:
        print(f"   📺 Channel: {SIGNAL_INFO['channel']}")
    if SIGNAL_INFO['band']:
        print(f"   📻 Band: {SIGNAL_INFO['band']}")
    
    SIGNAL_CATEGORY = categorize_signal(SIGNAL_INFO['rssi'], CONFIG['SIGNAL_THRESHOLD_DBM'])
    if SIGNAL_CATEGORY == "good_signal":
        SIGNAL_DISPLAY = f"Good Signal ({SIGNAL_INFO['rssi']} dBm)"
        print(f"   ✅ Status: GOOD SIGNAL (>= {CONFIG['SIGNAL_THRESHOLD_DBM']} dBm)")
    elif SIGNAL_CATEGORY == "bad_signal":
        SIGNAL_DISPLAY = f"Bad Signal ({SIGNAL_INFO['rssi']} dBm)"
        print(f"   ⚠️ Status: BAD SIGNAL (< {CONFIG['SIGNAL_THRESHOLD_DBM']} dBm)")
    else:
        SIGNAL_DISPLAY = f"SSID: {SIGNAL_INFO['ssid']}" if SIGNAL_INFO['ssid'] else "Unknown"
        print(f"   ℹ️ Status: RSSI tidak terdeteksi")

# 2. Detect DNS
print("\n🌐 Mendeteksi DNS...")
DNS_INFO = get_dns_servers()

if DNS_INFO["error"]:
    print(f"   ⚠️ Warning: {DNS_INFO['error']}")
    if CONFIG['ONT_DNS'] != 'N/A':
        DNS_INFO["primary_dns"] = CONFIG['ONT_DNS']
        DNS_INFO["dns_servers"] = [CONFIG['ONT_DNS']]
        print(f"   📋 Menggunakan DNS dari config: {CONFIG['ONT_DNS']}")
else:
    print(f"   🌐 DNS Servers: {', '.join(DNS_INFO['dns_servers'])}")
    print(f"   🎯 Primary DNS: {DNS_INFO['primary_dns']}")

# 3. Detect Location
print("\n📍 Mendeteksi Lokasi...")
LOCATION_INFO = get_device_location()

if LOCATION_INFO["error"]:
    print(f"   ⚠️ Warning: {LOCATION_INFO['error']}")
else:
    print(f"   🌍 Lokasi: {LOCATION_INFO['city']}, {LOCATION_INFO['region']}, {LOCATION_INFO['country']}")
    print(f"   📍 Koordinat: {LOCATION_INFO['latitude']}, {LOCATION_INFO['longitude']}")
    if LOCATION_INFO.get('accuracy'):
        print(f"   🎯 Akurasi: ±{LOCATION_INFO['accuracy']:.1f}m")
    print(f"   🏢 ISP: {LOCATION_INFO['isp']}")
    print(f"   🌐 Public IP: {LOCATION_INFO['ip']}")
    print(f"   📡 Metode: {LOCATION_INFO['method']}")

# Build detection summary
BAND_LABEL = SIGNAL_INFO.get('band') or 'Unknown'
if BAND_LABEL and BAND_LABEL != 'Unknown':
    BAND_LABEL_SHORT = BAND_LABEL.replace('.', '_').replace('GHz', 'G')
else:
    BAND_LABEL_SHORT = "UnknownBand"

DNS_LABEL = DNS_INFO.get('primary_dns') or 'UnknownDNS'
if DNS_LABEL and DNS_LABEL != 'UnknownDNS':
    DNS_LABEL_SHORT = DNS_LABEL.replace('.', '-')
else:
    DNS_LABEL_SHORT = "UnknownDNS"

print("\n" + "=" * 60)
print("📋 RINGKASAN DETEKSI")
print("=" * 60)
print(f"   📶 Signal: {SIGNAL_DISPLAY}")
print(f"   📻 Band: {BAND_LABEL}")
print(f"   🌐 DNS: {DNS_INFO.get('primary_dns', 'Unknown')}")
if LOCATION_INFO.get('city'):
    print(f"   📍 Lokasi: {LOCATION_INFO['city']}, {LOCATION_INFO['country']}")
if LOCATION_INFO.get('accuracy'):
    print(f"   🎯 Akurasi Lokasi: ±{LOCATION_INFO['accuracy']:.1f}m")
print(f"   🏷️ Session Label: {SIGNAL_CATEGORY}_{BAND_LABEL_SHORT}_{DNS_LABEL_SHORT}")
print("=" * 60)

# Tip for precise location
if not PRECISE_LOCATION_FILE.exists():
    print("\n💡 Tip: Untuk lokasi GPS yang lebih akurat, jalankan:")
    print("   python get_location.py")

🔍 AUTO-DETECT KONDISI JARINGAN

📶 Mendeteksi WiFi...
   📡 SSID: None
   📶 RSSI: -44 dBm
   📺 Channel: 149
   📻 Band: 5GHz
   ✅ Status: GOOD SIGNAL (>= -65 dBm)

🌐 Mendeteksi DNS...
   🌐 DNS Servers: 192.168.111.1
   🎯 Primary DNS: 192.168.111.1

📍 Mendeteksi Lokasi...
   🌍 Lokasi: Jakarta, Jakarta, Indonesia
   📍 Koordinat: -6.18104, 106.826
   🏢 ISP: PT. TELKOM INDONESIA
   🌐 Public IP: 118.97.180.34
   📡 Metode: IP Geolocation

📋 RINGKASAN DETEKSI
   📶 Signal: Good Signal (-44 dBm)
   📻 Band: 5GHz
   🌐 DNS: 192.168.111.1
   📍 Lokasi: Jakarta, Indonesia
   🏷️ Session Label: good_signal_5G_192-168-111-1


In [7]:
# ============================================
# CELL 4: HELPER FUNCTIONS
# ============================================
from tqdm.notebook import tqdm

def run_ping_test(host: str = "8.8.8.8", duration: int = 60) -> dict:
    """Run ping test for specified duration and return statistics."""
    print(f"🏓 Running ping to {host} for {duration} seconds...")
    
    system = platform.system()
    
    if system == "Windows":
        count = duration
        cmd = ["ping", "-n", str(count), host]
    else:
        count = duration
        cmd = ["ping", "-c", str(count), host]
    
    result = {
        "host": host,
        "duration": duration,
        "packets_sent": None,
        "packets_received": None,
        "packets_lost": None,
        "loss_percent": None,
        "min_ms": None,
        "max_ms": None,
        "avg_ms": None,
        "raw_output": None,
        "error": None
    }
    
    try:
        proc = subprocess.run(cmd, capture_output=True, text=True, timeout=duration + 30)
        result["raw_output"] = proc.stdout
        output = proc.stdout
        
        if system == "Windows":
            match = re.search(r'Sent = (\d+), Received = (\d+), Lost = (\d+) \((\d+)% loss\)', output)
            if match:
                result["packets_sent"] = int(match.group(1))
                result["packets_received"] = int(match.group(2))
                result["packets_lost"] = int(match.group(3))
                result["loss_percent"] = int(match.group(4))
            
            match = re.search(r'Minimum = (\d+)ms, Maximum = (\d+)ms, Average = (\d+)ms', output)
            if match:
                result["min_ms"] = int(match.group(1))
                result["max_ms"] = int(match.group(2))
                result["avg_ms"] = int(match.group(3))
        else:
            match = re.search(r'(\d+) packets transmitted, (\d+) (?:packets )?received, (\d+(?:\.\d+)?)% packet loss', output)
            if match:
                result["packets_sent"] = int(match.group(1))
                result["packets_received"] = int(match.group(2))
                result["loss_percent"] = float(match.group(3))
                result["packets_lost"] = result["packets_sent"] - result["packets_received"]
            
            match = re.search(r'min/avg/max/(?:mdev|stddev) = ([\d.]+)/([\d.]+)/([\d.]+)', output)
            if match:
                result["min_ms"] = float(match.group(1))
                result["avg_ms"] = float(match.group(2))
                result["max_ms"] = float(match.group(3))
        
        print(f"   ✅ Packets: {result['packets_sent']} sent, {result['packets_received']} received, {result['loss_percent']}% loss")
        print(f"   ✅ RTT: min={result['min_ms']}ms, avg={result['avg_ms']}ms, max={result['max_ms']}ms")
        
    except subprocess.TimeoutExpired:
        result["error"] = "Timeout"
        print(f"   ❌ Ping timeout")
    except Exception as e:
        result["error"] = str(e)
        print(f"   ❌ Error: {e}")
    
    return result


def measure_ttfb_curl(url: str) -> dict:
    """Measure TTFB using curl with detailed timing breakdown."""
    timestamp = datetime.now().isoformat()
    
    curl_format = (
        '{"time_namelookup": %{time_namelookup}, '
        '"time_connect": %{time_connect}, '
        '"time_appconnect": %{time_appconnect}, '
        '"time_starttransfer": %{time_starttransfer}, '
        '"time_total": %{time_total}, '
        '"http_code": %{http_code}}'
    )
    
    cmd = [
        "curl",
        "-o", "/dev/null" if platform.system() != "Windows" else "NUL",
        "-s",
        "-w", curl_format,
        "--max-time", "30",
        url
    ]
    
    result = {
        "timestamp": timestamp,
        "url": url,
        "lookup_ms": None,
        "connect_ms": None,
        "appconnect_ms": None,
        "ttfb_ms": None,
        "total_ms": None,
        "tcp_rtt_ms": None,
        "tls_ms": None,
        "server_response_ms": None,
        "http_code": None,
        "error": None,
        "status": "unknown"
    }
    
    try:
        proc = subprocess.run(cmd, capture_output=True, text=True, timeout=35)
        
        if proc.returncode == 0:
            data = json.loads(proc.stdout)
            
            result["lookup_ms"] = round(data["time_namelookup"] * 1000, 2)
            result["connect_ms"] = round(data["time_connect"] * 1000, 2)
            result["appconnect_ms"] = round(data["time_appconnect"] * 1000, 2)
            result["ttfb_ms"] = round(data["time_starttransfer"] * 1000, 2)
            result["total_ms"] = round(data["time_total"] * 1000, 2)
            result["http_code"] = data["http_code"]
            
            result["tcp_rtt_ms"] = round(result["connect_ms"] - result["lookup_ms"], 2)
            result["tls_ms"] = round(result["appconnect_ms"] - result["connect_ms"], 2)
            result["server_response_ms"] = round(result["ttfb_ms"] - result["appconnect_ms"], 2)
            
            if result["ttfb_ms"] < CONFIG['TTFB_GOOD_MS']:
                result["status"] = "good"
            elif result["ttfb_ms"] < CONFIG['TTFB_WARNING_MS']:
                result["status"] = "warning"
            else:
                result["status"] = "poor"
        else:
            result["error"] = proc.stderr.strip() or f"Exit code: {proc.returncode}"
            result["status"] = "error"
            
    except subprocess.TimeoutExpired:
        result["error"] = "Timeout (>30s)"
        result["status"] = "timeout"
    except Exception as e:
        result["error"] = str(e)
        result["status"] = "error"
    
    return result


def run_batch_ttfb_test(url: str, sample_count: int, delay_seconds: int, target_name: str, scenario: str) -> pd.DataFrame:
    """Run batch curl TTFB tests with delay between each test."""
    results = []
    
    print(f"\n🚀 Testing: {target_name}")
    print(f"   URL: {url}")
    print(f"   Samples: {sample_count}, Delay: {delay_seconds}s")
    print("-" * 50)
    
    with tqdm(total=sample_count, desc=target_name[:20]) as pbar:
        for i in range(1, sample_count + 1):
            result = measure_ttfb_curl(url)
            result["sample_num"] = i
            result["target_name"] = target_name
            result["scenario"] = scenario
            results.append(result)
            
            pbar.set_postfix({
                "TTFB": f"{result['ttfb_ms']}ms" if result['ttfb_ms'] else "ERR",
                "status": result["status"]
            })
            pbar.update(1)
            
            if i < sample_count:
                time.sleep(delay_seconds)
    
    df = pd.DataFrame(results)
    
    # Print summary
    valid_df = df[df['ttfb_ms'].notna()]
    if not valid_df.empty:
        print(f"📊 Summary: Mean={valid_df['ttfb_ms'].mean():.0f}ms, "
              f"Median={valid_df['ttfb_ms'].median():.0f}ms, "
              f"Min={valid_df['ttfb_ms'].min():.0f}ms, "
              f"Max={valid_df['ttfb_ms'].max():.0f}ms")
    
    return df


print("✅ Helper functions loaded")

✅ Helper functions loaded


In [ ]:
# ============================================
# CELL 5: CREATE SESSION & RUN TESTS
# ============================================

# Create session directory with informative naming
RESULTS_DIR = Path("./results")
RESULTS_DIR.mkdir(exist_ok=True)

SESSION_ID = datetime.now().strftime("%Y%m%d_%H%M%S")

# Build informative session folder name
# Format: session_{signal}_{band}_{dns}_{timestamp}
# Example: session_good_signal_5G_8-8-8-8_20260413_123456
SESSION_DIR = RESULTS_DIR / f"session_{SIGNAL_CATEGORY}_{BAND_LABEL_SHORT}_{DNS_LABEL_SHORT}_{SESSION_ID}"
SESSION_DIR.mkdir(exist_ok=True)

print("=" * 70)
print("🚀 MEMULAI TEST")
print("=" * 70)
print(f"📁 Session: {SESSION_DIR.name}")
print(f"\n📋 Kondisi Terdeteksi:")
print(f"   📶 Signal: {SIGNAL_DISPLAY} [{SIGNAL_CATEGORY}]")
print(f"   📻 Band: {BAND_LABEL}")
print(f"   🌐 DNS: {DNS_INFO.get('primary_dns', 'Unknown')}")
print(f"   📡 SSID: {SIGNAL_INFO.get('ssid', 'N/A')}")
if LOCATION_INFO.get('city'):
    print(f"   📍 Lokasi: {LOCATION_INFO['city']}, {LOCATION_INFO['country']}")
    print(f"   🌐 ISP: {LOCATION_INFO.get('isp', 'N/A')}")
print(f"\n🎯 Targets: {len(CONFIG['TARGETS'])}")
print(f"📊 Samples per target: {CONFIG['SAMPLE_COUNT']}")
print("=" * 70)

# Save comprehensive session info
session_info = {
    "session_id": SESSION_ID,
    "session_folder": SESSION_DIR.name,
    "start_time": datetime.now().isoformat(),
    "system": platform.system(),
    "detected_conditions": {
        "signal_category": SIGNAL_CATEGORY,
        "signal_rssi_dbm": SIGNAL_INFO.get("rssi"),
        "signal_threshold_dbm": CONFIG['SIGNAL_THRESHOLD_DBM'],
        "wifi_ssid": SIGNAL_INFO.get("ssid"),
        "wifi_channel": SIGNAL_INFO.get("channel"),
        "wifi_band": SIGNAL_INFO.get("band"),
        "dns_primary": DNS_INFO.get("primary_dns"),
        "dns_servers": DNS_INFO.get("dns_servers", [])
    },
    "location": {
        "latitude": LOCATION_INFO.get("latitude"),
        "longitude": LOCATION_INFO.get("longitude"),
        "city": LOCATION_INFO.get("city"),
        "region": LOCATION_INFO.get("region"),
        "country": LOCATION_INFO.get("country"),
        "isp": LOCATION_INFO.get("isp"),
        "public_ip": LOCATION_INFO.get("ip"),
        "detection_method": LOCATION_INFO.get("method")
    },
    "config": {
        "targets": CONFIG['TARGETS'],
        "sample_count": CONFIG['SAMPLE_COUNT'],
        "delay_seconds": CONFIG['DELAY_SECONDS'],
        "ping_duration": CONFIG['PING_DURATION'],
        "ttfb_good_ms": CONFIG['TTFB_GOOD_MS'],
        "ttfb_warning_ms": CONFIG['TTFB_WARNING_MS']
    }
}

with open(SESSION_DIR / "session_info.json", "w") as f:
    json.dump(session_info, f, indent=2)
print("💾 Session info saved")

# Run ping test first
print("\n" + "=" * 70)
print("📍 PING TEST")
print("=" * 70)
ping_result = run_ping_test("8.8.8.8", CONFIG['PING_DURATION'])

# Save ping result with detailed info
ping_filename = f"ping_result_{BAND_LABEL_SHORT}_{DNS_LABEL_SHORT}.txt"
with open(SESSION_DIR / ping_filename, "w") as f:
    f.write(f"Ping Test Results\n")
    f.write("=" * 60 + "\n\n")
    f.write(f"Session: {SESSION_DIR.name}\n")
    f.write(f"Signal: {SIGNAL_CATEGORY} ({SIGNAL_INFO.get('rssi', 'N/A')} dBm)\n")
    f.write(f"Band: {BAND_LABEL}\n")
    f.write(f"DNS: {DNS_INFO.get('primary_dns', 'N/A')}\n")
    if LOCATION_INFO.get('city'):
        f.write(f"Location: {LOCATION_INFO['city']}, {LOCATION_INFO['country']}\n")
        f.write(f"Coordinates: {LOCATION_INFO['latitude']}, {LOCATION_INFO['longitude']}\n")
        f.write(f"ISP: {LOCATION_INFO.get('isp', 'N/A')}\n")
    f.write(f"\nHost: {ping_result['host']}\n")
    f.write(f"Packets: {ping_result['packets_sent']} sent, {ping_result['packets_received']} received\n")
    f.write(f"Loss: {ping_result['loss_percent']}%\n")
    f.write(f"RTT: min={ping_result['min_ms']}ms, avg={ping_result['avg_ms']}ms, max={ping_result['max_ms']}ms\n")
    f.write("\n" + "=" * 60 + "\n")
    f.write("Raw Output:\n")
    f.write(ping_result['raw_output'] or "N/A")

print(f"💾 Ping result saved: {ping_filename}")

# Run TTFB tests for all targets
print("\n" + "=" * 70)
print("📍 TTFB TESTS")
print("=" * 70)

all_results = []

for i, target_url in enumerate(CONFIG['TARGETS'], 1):
    domain = urlparse(target_url).netloc
    # Informative target name
    target_name = f"Target_{i}_{domain}"
    
    df = run_batch_ttfb_test(
        target_url,
        CONFIG['SAMPLE_COUNT'],
        CONFIG['DELAY_SECONDS'],
        target_name,
        SIGNAL_CATEGORY
    )
    
    # Add detection info to each row
    df['wifi_band'] = BAND_LABEL
    df['wifi_rssi'] = SIGNAL_INFO.get('rssi')
    df['dns_primary'] = DNS_INFO.get('primary_dns')
    df['signal_category'] = SIGNAL_CATEGORY
    df['location_city'] = LOCATION_INFO.get('city')
    df['location_country'] = LOCATION_INFO.get('country')
    df['isp'] = LOCATION_INFO.get('isp')
    
    all_results.append(df)
    
    # Save individual target results with informative name
    csv_filename = f"{target_name}_{BAND_LABEL_SHORT}.csv"
    df.to_csv(SESSION_DIR / csv_filename, index=False)
    print(f"💾 Saved: {csv_filename}")

# Combine all results
combined_df = pd.concat(all_results, ignore_index=True)

# Save combined results with informative name
combined_filename = f"all_results_{SIGNAL_CATEGORY}_{BAND_LABEL_SHORT}_{DNS_LABEL_SHORT}.csv"
combined_df.to_csv(SESSION_DIR / combined_filename, index=False)
print(f"\n💾 Combined results saved: {combined_filename}")

print("\n" + "=" * 70)
print("✅ SEMUA TEST SELESAI")
print("=" * 70)

🚀 MEMULAI TEST
📁 Session: session_good_signal_5G_192-168-111-1_20260413_114500

📋 Kondisi Terdeteksi:
   📶 Signal: Good Signal (-44 dBm) [good_signal]
   📻 Band: 5GHz
   🌐 DNS: 192.168.111.1
   📡 SSID: None
   📍 Lokasi: Jakarta, Indonesia
   🌐 ISP: PT. TELKOM INDONESIA

🎯 Targets: 2
📊 Samples per target: 10
💾 Session info saved

📍 PING TEST
🏓 Running ping to 8.8.8.8 for 60 seconds...
   ✅ Packets: 60 sent, 60 received, 0.0% loss
   ✅ RTT: min=22.575ms, avg=35.519ms, max=159.359ms
💾 Ping result saved: ping_result_5G_192-168-111-1.txt

📍 TTFB TESTS

🚀 Testing: Target_1_www.instagram.com
   URL: https://www.instagram.com
   Samples: 10, Delay: 30s
--------------------------------------------------


Target_1_www.instagr:   0%|          | 0/10 [00:00<?, ?it/s]

In [ ]:
# ============================================
# CELL 6: ADAPTIVE ANALYSIS & VISUALIZATION
# ============================================
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_palette("husl")

print("=" * 70)
print("📊 ANALISIS HASIL (ADAPTIVE REPORT)")
print("=" * 70)

# Summary statistics
summary = combined_df.groupby('target_name').agg({
    'ttfb_ms': ['count', 'mean', 'median', 'min', 'max', 'std']
}).round(2)
summary.columns = ['Count', 'Mean (ms)', 'Median (ms)', 'Min (ms)', 'Max (ms)', 'Std (ms)']

summary['Status'] = summary['Mean (ms)'].apply(
    lambda x: '✅ GOOD' if x < CONFIG['TTFB_GOOD_MS'] else ('⚠️ WARNING' if x < CONFIG['TTFB_WARNING_MS'] else '❌ POOR')
)

print("\n📈 TTFB Summary per Target:")
print("-" * 80)
print(summary.to_string())
print()

# Save summary with informative name
summary_filename = f"summary_{SIGNAL_CATEGORY}_{BAND_LABEL_SHORT}_{DNS_LABEL_SHORT}.csv"
summary.to_csv(SESSION_DIR / summary_filename)
print(f"💾 Summary saved: {summary_filename}")

# ==== ADAPTIVE CHART TITLE ====
signal_emoji = "✅" if SIGNAL_CATEGORY == "good_signal" else ("⚠️" if SIGNAL_CATEGORY == "bad_signal" else "❓")

# Build location string for title
location_str = ""
if LOCATION_INFO.get('city'):
    location_str = f" | 📍 {LOCATION_INFO['city']}"

chart_title = (
    f'NOC Tune - TTFB Analysis\n'
    f'{signal_emoji} {SIGNAL_DISPLAY} | {BAND_LABEL} | DNS: {DNS_INFO.get("primary_dns", "N/A")}{location_str}'
)

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(chart_title, fontsize=14, fontweight='bold')

# 1. Boxplot by target
ax1 = axes[0, 0]
combined_df.boxplot(column='ttfb_ms', by='target_name', ax=ax1)
ax1.set_title('TTFB Distribution by Target')
ax1.set_xlabel('Target')
ax1.set_ylabel('TTFB (ms)')
ax1.axhline(y=CONFIG['TTFB_GOOD_MS'], color='green', linestyle='--', label=f"Good (<{CONFIG['TTFB_GOOD_MS']}ms)")
ax1.axhline(y=CONFIG['TTFB_WARNING_MS'], color='orange', linestyle='--', label=f"Warning (<{CONFIG['TTFB_WARNING_MS']}ms)")
ax1.legend(fontsize=8)
ax1.tick_params(axis='x', rotation=45)
plt.suptitle('')

# 2. Mean TTFB bar chart
ax2 = axes[0, 1]
means = combined_df.groupby('target_name')['ttfb_ms'].mean().sort_values()
colors = ['green' if m < CONFIG['TTFB_GOOD_MS'] else 'orange' if m < CONFIG['TTFB_WARNING_MS'] else 'red' for m in means.values]
bars = ax2.barh(range(len(means)), means.values, color=colors)
ax2.set_yticks(range(len(means)))
ax2.set_yticklabels([n.split('_')[-1][:20] for n in means.index])
ax2.set_xlabel('Mean TTFB (ms)')
ax2.set_title('Mean TTFB Comparison')
ax2.axvline(x=CONFIG['TTFB_GOOD_MS'], color='green', linestyle='--', alpha=0.7)
ax2.axvline(x=CONFIG['TTFB_WARNING_MS'], color='orange', linestyle='--', alpha=0.7)
for bar, val in zip(bars, means.values):
    ax2.text(val + 10, bar.get_y() + bar.get_height()/2, f'{val:.0f}', va='center', fontsize=9)

# 3. TTFB over time (line plot)
ax3 = axes[1, 0]
for target in combined_df['target_name'].unique():
    target_df = combined_df[combined_df['target_name'] == target]
    ax3.plot(target_df['sample_num'], target_df['ttfb_ms'], marker='o', label=target.split('_')[-1][:15])
ax3.set_xlabel('Sample Number')
ax3.set_ylabel('TTFB (ms)')
ax3.set_title('TTFB Over Time')
ax3.axhline(y=CONFIG['TTFB_GOOD_MS'], color='green', linestyle='--', alpha=0.5)
ax3.axhline(y=CONFIG['TTFB_WARNING_MS'], color='orange', linestyle='--', alpha=0.5)
ax3.legend(fontsize=8, loc='upper right')

# 4. Status distribution pie chart
ax4 = axes[1, 1]
status_counts = combined_df['status'].value_counts()
colors_pie = {'good': 'green', 'warning': 'orange', 'poor': 'red', 'error': 'gray', 'timeout': 'darkgray'}
ax4.pie(status_counts.values, labels=status_counts.index, autopct='%1.1f%%',
        colors=[colors_pie.get(s, 'blue') for s in status_counts.index])
ax4.set_title('TTFB Status Distribution')

plt.tight_layout()

# Save chart with informative name
chart_filename = f"analysis_chart_{SIGNAL_CATEGORY}_{BAND_LABEL_SHORT}_{DNS_LABEL_SHORT}.png"
fig.savefig(SESSION_DIR / chart_filename, dpi=150, bbox_inches='tight')
print(f"💾 Chart saved: {chart_filename}")

plt.show()

In [ ]:
# ============================================
# CELL 7: ADAPTIVE CONCLUSION & RECOMMENDATIONS
# ============================================

print("\n" + "=" * 70)
print("📋 KESIMPULAN (ADAPTIVE REPORT)")
print("=" * 70)

# ==== Detection Summary ====
print("\n🔍 KONDISI TERDETEKSI:")
print("-" * 50)

# Signal status
if SIGNAL_CATEGORY == "good_signal":
    signal_status = "BAGUS ✅"
elif SIGNAL_CATEGORY == "bad_signal":
    signal_status = "JELEK ⚠️"
else:
    signal_status = "TIDAK DIKETAHUI ❓"

print(f"   📶 Signal: {signal_status}")
print(f"      • RSSI: {SIGNAL_INFO.get('rssi', 'N/A')} dBm")
print(f"      • Threshold: {CONFIG['SIGNAL_THRESHOLD_DBM']} dBm")
print(f"   📻 Band: {BAND_LABEL}")
print(f"      • Channel: {SIGNAL_INFO.get('channel', 'N/A')}")
print(f"   🌐 DNS: {DNS_INFO.get('primary_dns', 'N/A')}")
if len(DNS_INFO.get('dns_servers', [])) > 1:
    print(f"      • All servers: {', '.join(DNS_INFO['dns_servers'])}")
print(f"   📡 SSID: {SIGNAL_INFO.get('ssid', 'N/A')}")

# Location info
if LOCATION_INFO.get('city'):
    print(f"\n📍 LOKASI TEST:")
    print("-" * 50)
    print(f"   🌍 {LOCATION_INFO['city']}, {LOCATION_INFO.get('region', '')}, {LOCATION_INFO['country']}")
    print(f"   📍 Koordinat: {LOCATION_INFO['latitude']}, {LOCATION_INFO['longitude']}")
    if LOCATION_INFO.get('accuracy'):
        print(f"   🎯 Akurasi: ±{LOCATION_INFO['accuracy']:.1f}m")
    print(f"   🏢 ISP: {LOCATION_INFO.get('isp', 'N/A')}")
    print(f"   🌐 Public IP: {LOCATION_INFO.get('ip', 'N/A')}")
    print(f"   📡 Metode: {LOCATION_INFO.get('method', 'N/A')}")
    
    # Tip for precise location
    if not LOCATION_INFO.get('accuracy'):
        print(f"\n   💡 Untuk GPS presisi, jalankan: python get_location.py")

# ==== TTFB Summary ====
print(f"\n📊 HASIL TTFB:")
print("-" * 50)
print(f"   Total tests: {len(combined_df)}")

valid_df = combined_df[combined_df['ttfb_ms'].notna()]
if not valid_df.empty:
    overall_mean = valid_df['ttfb_ms'].mean()
    overall_median = valid_df['ttfb_ms'].median()
    overall_min = valid_df['ttfb_ms'].min()
    overall_max = valid_df['ttfb_ms'].max()
    overall_std = valid_df['ttfb_ms'].std()
    
    if overall_mean < CONFIG['TTFB_GOOD_MS']:
        ttfb_status = '✅ GOOD'
        ttfb_verdict = 'Performa jaringan BAIK'
    elif overall_mean < CONFIG['TTFB_WARNING_MS']:
        ttfb_status = '⚠️ WARNING'
        ttfb_verdict = 'Performa jaringan CUKUP, perlu monitoring'
    else:
        ttfb_status = '❌ POOR'
        ttfb_verdict = 'Performa jaringan BURUK, perlu investigasi'
    
    print(f"   🎯 Mean TTFB: {overall_mean:.2f} ms {ttfb_status}")
    print(f"   📊 Median: {overall_median:.2f} ms")
    print(f"   📉 Min: {overall_min:.2f} ms")
    print(f"   📈 Max: {overall_max:.2f} ms")
    print(f"   📏 Std Dev: {overall_std:.2f} ms")
    
    # Status breakdown
    status_counts = valid_df['status'].value_counts()
    print(f"\n   Status Breakdown:")
    for status, count in status_counts.items():
        pct = (count / len(valid_df)) * 100
        icon = {'good': '✅', 'warning': '⚠️', 'poor': '❌', 'error': '💥', 'timeout': '⏱️'}.get(status, '❓')
        print(f"      {icon} {status}: {count} ({pct:.1f}%)")

# ==== Adaptive Recommendations ====
print(f"\n💡 REKOMENDASI:")
print("-" * 50)

recommendations = []

# Signal-based recommendations
if SIGNAL_CATEGORY == "bad_signal":
    recommendations.append("📶 Signal lemah terdeteksi. Pertimbangkan:")
    recommendations.append("   • Dekatkan perangkat ke router/access point")
    recommendations.append("   • Periksa interference dari perangkat lain")
    recommendations.append("   • Pertimbangkan menggunakan WiFi extender")
elif SIGNAL_CATEGORY == "unknown_signal":
    recommendations.append("📶 Signal tidak terdeteksi:")
    recommendations.append("   • Untuk deteksi RSSI di macOS, install: pip install pyobjc-framework-CoreWLAN")
    recommendations.append("   • Pastikan WiFi aktif dan terhubung")

# Band-based recommendations
if BAND_LABEL == "2.4GHz":
    recommendations.append("📻 Menggunakan band 2.4GHz:")
    recommendations.append("   • Band ini memiliki jangkauan lebih luas tapi kecepatan lebih rendah")
    recommendations.append("   • Lebih rentan interference dari perangkat lain")
    recommendations.append("   • Coba test dengan band 5GHz untuk perbandingan")
elif BAND_LABEL == "5GHz":
    recommendations.append("📻 Menggunakan band 5GHz:")
    recommendations.append("   • Band ini memiliki kecepatan lebih tinggi tapi jangkauan lebih pendek")
    recommendations.append("   • Coba test dengan band 2.4GHz untuk perbandingan jangkauan")

# DNS-based recommendations
primary_dns = DNS_INFO.get('primary_dns', '')
if primary_dns in ['8.8.8.8', '8.8.4.4']:
    recommendations.append("🌐 Menggunakan Google DNS:")
    recommendations.append("   • DNS publik umumnya cepat dan reliable")
    recommendations.append("   • Coba test dengan DNS ISP untuk perbandingan")
elif primary_dns and (primary_dns.startswith('192.168') or primary_dns.startswith('10.')):
    recommendations.append("🌐 Menggunakan DNS lokal/ISP:")
    recommendations.append("   • DNS lokal mungkin lebih cepat untuk konten lokal")
    recommendations.append("   • Coba test dengan DNS publik (8.8.8.8) untuk perbandingan")

# Location-based recommendations
if not LOCATION_INFO.get('accuracy'):
    recommendations.append("📍 Lokasi menggunakan IP Geolocation (~city level):")
    recommendations.append("   • Untuk lokasi GPS presisi, jalankan: python get_location.py")
    recommendations.append("   • Browser akan meminta izin lokasi untuk akurasi ±10m")

# TTFB-based recommendations
if valid_df is not None and not valid_df.empty:
    if overall_mean >= CONFIG['TTFB_WARNING_MS']:
        recommendations.append("⏱️ TTFB tinggi terdeteksi:")
        recommendations.append("   • Periksa beban jaringan (streaming, download)")
        recommendations.append("   • Restart router jika sudah lama tidak direstart")
        recommendations.append("   • Hubungi ISP jika masalah berlanjut")
    
    if overall_std > overall_mean * 0.5:  # High variance
        recommendations.append("📊 Variasi TTFB tinggi terdeteksi:")
        recommendations.append("   • Jaringan tidak stabil")
        recommendations.append("   • Mungkin ada congestion atau interference")

for rec in recommendations:
    print(f"   {rec}")

if not recommendations:
    print("   ✅ Tidak ada masalah signifikan yang terdeteksi!")

# ==== Next Steps ====
print(f"\n📝 LANGKAH SELANJUTNYA:")
print("-" * 50)
if BAND_LABEL == "5GHz":
    print("   1. Jalankan test dengan band 2.4GHz untuk perbandingan")
elif BAND_LABEL == "2.4GHz":
    print("   1. Jalankan test dengan band 5GHz untuk perbandingan")
else:
    print("   1. Identifikasi band WiFi yang digunakan (2.4GHz atau 5GHz)")

if SIGNAL_CATEGORY == "good_signal":
    print("   2. Jalankan test di area dengan sinyal lemah untuk perbandingan")
elif SIGNAL_CATEGORY == "bad_signal":
    print("   2. Jalankan test di area dengan sinyal kuat untuk perbandingan")
else:
    print("   2. Install pyobjc-framework-CoreWLAN untuk deteksi signal yang lebih baik")

print("   3. Bandingkan hasil dengan mengubah DNS (8.8.8.8 vs DNS ISP)")
print("   4. Jalankan test di waktu berbeda (pagi vs malam)")

# ==== Session Info ====
print(f"\n📁 FILE OUTPUT:")
print("-" * 50)
print(f"   📂 Folder: {SESSION_DIR}")
for f in sorted(SESSION_DIR.iterdir()):
    print(f"   📄 {f.name}")

print("\n" + "=" * 70)
print("✅ TEST SELESAI - Terima kasih telah menggunakan NOC Tune!")
print("=" * 70)